# Seizure Prediction using CNNs on CHB-MIT Dataset

This notebook consolidates the full pipeline:
1. **Data Processing** — EDF to Spectrogram conversion
2. **CNN Model** — Training and evaluation
3. **Threshold Testing** — Sensitivity and FPR analysis

---
## Part 1: Data Processing — EDF to Spectrogram

Converts raw EDF files from the CHB-MIT dataset into spectrograms for CNN input.

In [ ]:
!wget -r --no-parent https://archive.physionet.org/pn6/chbmit/

Streaming output truncated to the last 5000 lines.
Length: 4625 (4.5K)
Saving to: ‘archive.physionet.org/pn6/chbmit/chb15/SHA256SUMS’

archive.physionet.o 100%[===================>]   4.52K  --.-KB/s    in 0s      

2026-03-11 03:24:10 (80.2 MB/s) - ‘archive.physionet.org/pn6/chbmit/chb15/SHA256SUMS’ saved [4625/4625]

--2026-03-11 03:24:10--  https://archive.physionet.org/pn6/chbmit/chb15/chb15-summary.txt
Reusing existing connection to archive.physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 6890 (6.7K) [text/plain]
Saving to: ‘archive.physionet.org/pn6/chbmit/chb15/chb15-summary.txt’

archive.physionet.o 100%[===================>]   6.73K  --.-KB/s    in 0s      

2026-03-11 03:24:10 (118 MB/s) - ‘archive.physionet.org/pn6/chbmit/chb15/chb15-summary.txt’ saved [6890/6890]

--2026-03-11 03:24:10--  https://archive.physionet.org/pn6/chbmit/chb15/chb15_01.edf
Reusing existing connection to archive.physionet.org:443.
HTTP request sent, awaiting response... 200 O

In [ ]:
!pip install pyedflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 24.7 MB/s eta 0:00:00


In [ ]:
import pyedflib
import numpy as np
from scipy import signal
from scipy.signal import butter, lfilter
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os

### Constants & Configuration

In [ ]:
# DATASET: https://physionet.org/pn6/chbmit/
sampleRate = 256
pathDataSet = ''  # path of the dataset
FirstPartPathOutput = ''  # path where the spectogram will be saved
#patients = ["01", "02", "03", "05", "09", "10", "13", "14", "18", "19", "20", "21", "23"]
#nSeizure = [7, 3, 6, 5, 4, 6, 5, 5, 6, 3, 5, 4, 5]
patients = ["01", "02", "05", "19", "21", "23"]
_30_MINUTES_OF_DATA = 256 * 60 * 30
_MINUTES_OF_DATA_BETWEEN_PRE_AND_SEIZURE = 3  # In theory 5 like SPH but set to 3 to account for some seizures in the paper
_MINUTES_OF_PREICTAL = 30
_SIZE_WINDOW_IN_SECONDS = 30
_SIZE_WINDOW_SPECTOGRAM = _SIZE_WINDOW_IN_SECONDS * 256
nSpectogram = 0
signalsBlock = None
SecondPartPathOutput = ''
legendOfOutput = ''
isPreictal = ''

### Load Parameters from File

In [ ]:
def loadParametersFromFile(filePath):
    global pathDataSet
    global FirstPartPathOutput
    if(os.path.isfile(filePath)):
        with open(filePath, "r") as f:
                line=f.readline()
                if(line.split(":")[0]=="pathDataSet"):
                    pathDataSet=line.split(":")[1].strip()
                line=f.readline()
                if(line.split(":")[0]=="FirstPartPathOutput"):
                    FirstPartPathOutput=line.split(":")[1].strip()

### Signal Filters

Bandstop and highpass Butterworth filters for noise removal.

In [ ]:
# Bandstop filter
def butter_bandstop_filter(data, lowcut, highcut, fs, order):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq

    i, u = butter(order, [low, high], btype='bandstop')
    y = lfilter(i, u, data)
    return y

# Bandstop filter, highpass
def butter_highpass_filter(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = lfilter(b, a, data)
    return y

### Data Loading

Functions to load patient summary files and EDF signal data.

In [ ]:
# Create file pointer to the patient summary with the given index
def loadSummaryPatient(index):
    f = open(pathDataSet+'chb'+patients[index]+'/chb'+patients[index]+'-summary.txt', 'r')
    return f

# Load data of a patient (indexPatient). Data is taken from the file named fileOfData
# Returns a numpy array with the patient data from the file
def loadDataOfPatient(indexPatient, fileOfData):
    f = pyedflib.EdfReader(pathDataSet+'chb'+patients[indexPatient]+'/'+fileOfData)
    n = f.signals_in_file
    sigbufs = np.zeros((n, f.getNSamples()[0]))
    for i in np.arange(n):
        sigbufs[i, :] = f.readSignal(i)
    sigbufs=cleanData(sigbufs, indexPatient)
    return sigbufs

def cleanData(Data, indexPatient):
    if(patients[indexPatient] in ["19","21"]):
        Data=np.delete(Data, 22, axis=0)
        Data=np.delete(Data, 17, axis=0)
        Data=np.delete(Data, 12, axis=0)
        Data=np.delete(Data, 9, axis=0)
        Data=np.delete(Data, 4, axis=0)
    return Data

### Time Parsing

Converts time strings to datetime objects, handling edge cases with hours exceeding 24.

In [ ]:
# Convert a string indicating a time to a datetime object
# and clean dates that don't respect hour limits
def getTime(dateInString):
    time=0
    try:
        time = datetime.strptime(dateInString, '%H:%M:%S')
    except ValueError:
        dateInString=" "+dateInString
        if(' 24' in dateInString):
            dateInString = dateInString.replace(' 24', '23')
            time = datetime.strptime(dateInString, '%H:%M:%S')
            time += timedelta(hours=1)
        else:
            dateInString = dateInString.replace(' 25', '23')
            time = datetime.strptime(dateInString, '%H:%M:%S')
            time += timedelta(hours=2)
    return time

### Save Signals to Disk

In [ ]:
def saveSignalsOnDisk(signalsBlock, nSpectogram):
    global SecondPartPathOutput
    global FirstPartPathOutput
    global legendOfOutput
    global isPreictal

    if not os.path.exists(FirstPartPathOutput):
        os.makedirs(FirstPartPathOutput)
    if not os.path.exists(FirstPartPathOutput+SecondPartPathOutput):
        os.makedirs(FirstPartPathOutput+SecondPartPathOutput)
    np.save(FirstPartPathOutput+SecondPartPathOutput+'/spec_'+isPreictal+'_'+str(nSpectogram-signalsBlock.shape[0])+'_'+str(nSpectogram-1), signalsBlock)
    legendOfOutput=legendOfOutput+str(nSpectogram-signalsBlock.shape[0])+' '+str(nSpectogram-1) +' '+SecondPartPathOutput+'/spec_'+isPreictal+'_'+str(nSpectogram-signalsBlock.shape[0])+'_'+str(nSpectogram-1) +'.npy\\n'

### Spectrogram Creation

Divides data into windows and creates spectrograms saved to disk.
- `S` is the factor indicating how much each window shifts
- Returns unconsidered data when data is not divisible by window length

In [ ]:
def createSpectrogram(data, S=0):
    global nSpectogram
    global signalsBlock
    global inB
    signals=np.zeros((22,59,114))

    t=0
    movement=int(S*256)
    if(S==0):
        movement=_SIZE_WINDOW_SPECTOGRAM
    while data.shape[1]-(t*movement+_SIZE_WINDOW_SPECTOGRAM) > 0:
        # SPECTROGRAM CREATION FOR ALL CHANNELS
        for i in range(0, 22):
            start = t*movement
            stop = start+_SIZE_WINDOW_SPECTOGRAM
            signals[i,:]=createSpec(data[i,start:stop])
        if(signalsBlock is None):
            signalsBlock=np.array([signals])
        else:
            signalsBlock=np.append(signalsBlock, [signals], axis=0)
        nSpectogram=nSpectogram+1
        if(signalsBlock.shape[0]==50):
            saveSignalsOnDisk(signalsBlock, nSpectogram)
            signalsBlock=None
            # SAVE SIGNALS
        t = t+1
    return (data.shape[1]-t*_SIZE_WINDOW_SPECTOGRAM)*-1

### Core Spectrogram Function

Applies bandstop and highpass filters, computes spectrogram, removes noise bands, and normalizes.

In [ ]:
# Function for the actual spectrogram creation.
def createSpec(data):
    fs=256
    lowcut=117
    highcut=123

    y=butter_bandstop_filter(data, lowcut, highcut, fs, order=6)
    lowcut=57
    highcut=63
    y=butter_bandstop_filter(y, lowcut, highcut, fs, order=6)

    cutoff=1
    y=butter_highpass_filter(y, cutoff, fs, order=6)

    Pxx=signal.spectrogram(y, nfft=256, fs=256, return_onesided=True, noverlap=128)[2]
    Pxx = np.delete(Pxx, np.s_[117:123+1], axis=0)
    Pxx = np.delete(Pxx, np.s_[57:63+1], axis=0)
    Pxx = np.delete(Pxx, 0, axis=0)

    result=(10*np.log10(np.transpose(Pxx))-(10*np.log10(np.transpose(Pxx))).min())/(np.max(10*np.log10(np.transpose(Pxx)))-np.min(10*np.log10(np.transpose(Pxx))))
    return result

### Spectrogram Creation with Visualization

In [ ]:
# Create spectrogram and display with matplotlib
def createSpecAndPlot(data):
    freqs, bins,Pxx =signal.spectrogram(data, nfft=256, fs=256, return_onesided=True, noverlap=128)

    print("Original")
    plt.pcolormesh(freqs, bins, 10*np.log10(np.transpose(Pxx)),cmap=plt.cm.jet)
    plt.colorbar()
    plt.ylabel('sec')
    plt.xlabel('Hz')
    plt.title('Spectrogram')
    plt.show()
    plt.close()


    fs=256
    lowcut=117
    highcut=123

    y=butter_bandstop_filter(data, lowcut, highcut, fs, order=6)
    lowcut=57
    highcut=63
    y=butter_bandstop_filter(y, lowcut, highcut, fs, order=6)

    cutoff=1
    y=butter_highpass_filter(y, cutoff, fs, order=6)

    #Pxx=signal.spectrogram(y, nfft=256, fs=256, return_onesided=True, noverlap=128)[2]
    freqs, bins,Pxx =signal.spectrogram(y, nfft=256, fs=256, return_onesided=True, noverlap=128)

    print("Filtered")
    plt.pcolormesh(freqs, bins, 10*np.log10(np.transpose(Pxx)),cmap=plt.cm.jet)
    plt.colorbar()
    plt.ylabel('sec')
    plt.xlabel('Hz')
    plt.title('Spectrogram')
    plt.show()
    plt.close()


    Pxx = np.delete(Pxx, np.s_[117:123+1], axis=0)
    Pxx = np.delete(Pxx, np.s_[57:63+1], axis=0)
    Pxx = np.delete(Pxx, 0, axis=0)

    print("Cleaned but not standard")
    freqs = np.arange(Pxx.shape[0])
    plt.pcolormesh(freqs, bins, 10*np.log10(np.transpose(Pxx)),cmap=plt.cm.jet)
    plt.colorbar()
    plt.ylabel('sec')
    plt.xlabel('Hz')
    plt.title('Spectrogram')
    plt.show()
    plt.close()

    result=(10*np.log10(np.transpose(Pxx))-(10*np.log10(np.transpose(Pxx))).min())/(np.max(10*np.log10(np.transpose(Pxx)))-np.min(10*np.log10(np.transpose(Pxx))))

    print("Standard")
    freqs = np.arange(result.shape[1])
    plt.pcolormesh(freqs, bins, result,cmap=plt.cm.jet)
    plt.colorbar()
    plt.ylabel('sec')
    plt.xlabel('Hz')
    plt.title('Spectrogram')
    plt.show()
    plt.close()

    return result

### Data Classes

Helper classes to represent data intervals and file metadata.

In [ ]:
# Class used to represent data intervals, both Preictal and Interictal
class PreIntData:
    start=0
    end=0
    def __init__(self, s, e):
        self.start=s
        self.end=e

# Class used to hold file data, start/end date-time and associated file name
class FileData:
    start=0
    end=0
    nameFile=""
    def __init__(self, s, e, nF):
        self.start=s
        self.end=e
        self.nameFile=nF

### Create Interval Arrays from Patient Summary

Loads all useful data from the analyzed patient's summary file.
Returns preictal intervals, interictal intervals, and file data arrays.

In [ ]:
def createArrayIntervalData(fSummary):
    preictalInteval=[]
    interictalInterval=[]
    interictalInterval.append(PreIntData(datetime.min, datetime.max))
    files=[]
    firstTime=True
    oldTime=datetime.min
    startTime=0
    line=fSummary.readline()
    endS=datetime.min
    while(line):
        data=line.split(':')
        if(data[0]=="File Name"):
            nF=data[1].strip()
            s=getTime((fSummary.readline().split(": "))[1].strip())
            if(firstTime):
                interictalInterval[0].start=s
                firstTime=False
                startTime=s
            while s<oldTime:
                s=s+ timedelta(hours=24)
            oldTime=s
            endTimeFile=getTime((fSummary.readline().split(": "))[1].strip())
            while endTimeFile<oldTime:
                endTimeFile=endTimeFile+ timedelta(hours=24)
            oldTime=endTimeFile
            files.append(FileData(s, endTimeFile,nF))
            for j in range(0, int((fSummary.readline()).split(':')[1])):
                secSt=int(fSummary.readline().split(': ')[1].split(' ')[0])
                secEn=int(fSummary.readline().split(': ')[1].split(' ')[0])
                ss=s+timedelta(seconds=secSt)- timedelta(minutes=_MINUTES_OF_DATA_BETWEEN_PRE_AND_SEIZURE+_MINUTES_OF_PREICTAL)
                if((len(preictalInteval)==0 or ss > endS) and ss-startTime>timedelta(minutes=20)):
                    ee=ss+ timedelta(minutes=_MINUTES_OF_PREICTAL)
                    preictalInteval.append(PreIntData(ss,ee))
                endS=s+timedelta(seconds=secEn)
                ss=s+timedelta(seconds=secSt)- timedelta(hours=4)
                ee=s+timedelta(seconds=secEn)+ timedelta(hours=4)
                if(interictalInterval[len(interictalInterval)-1].start<ss and interictalInterval[len(interictalInterval)-1].end>ee):
                    interictalInterval[len(interictalInterval)-1].end=ss
                    interictalInterval.append(PreIntData(ee, datetime.max))
                else:
                    if(interictalInterval[len(interictalInterval)-1].start<ee):
                        interictalInterval[len(interictalInterval)-1].start=ee
        line=fSummary.readline()
    fSummary.close()
    interictalInterval[len(interictalInterval)-1].end=endTimeFile
    return preictalInteval, interictalInterval, files

### Run Data Processing Pipeline

In [ ]:
def main_data_processing():
    global SecondPartPathOutput
    global FirstPartPathOutput
    global legendOfOutput
    global nSpectogram
    global signalsBlock
    global isPreictal
    print("START \\n")
    loadParametersFromFile("PARAMETERS_DATA_EDITING.txt")
    print("Parameters loaded")

    for indexPatient in range(0, len(patients)):
        print("Working on patient "+patients[indexPatient])
        legendOfOutput=""
        allLegend=""
        nSpectogram=0

        SecondPartPathOutput='/paz'+patients[indexPatient]
        f = loadSummaryPatient(indexPatient)
        preictalInfo, interictalInfo, filesInfo=createArrayIntervalData(f)
        if(patients[indexPatient]=="19"):
            preictalInfo.pop(0) # Remove first seizure data as it is not considered
        print("Summary patient loaded")

        # START interictal data management loop
        print("START creation interictal spectrogram")
        totInst=0
        #c=0
        #d=0
        interictalData = np.array([]).reshape(22,0)
        indexInterictalSegment=0
        isPreictal=''
        for fInfo in filesInfo:
            fileS=fInfo.start
            fileE=fInfo.end
            intSegStart=interictalInfo[indexInterictalSegment].start
            intSegEnd=interictalInfo[indexInterictalSegment].end
            while(fileS>intSegEnd and indexInterictalSegment<len(interictalInfo)):
                indexInterictalSegment=indexInterictalSegment+1
                intSegStart=interictalInfo[indexInterictalSegment].start
                intSegEnd=interictalInfo[indexInterictalSegment].end
            start=0
            end=0
            if(not fileE<intSegStart or fileS>intSegEnd):
                if(fileS>=intSegStart):
                    start=0
                else:
                    start=(intSegStart-fileS).seconds
                if(fileE<=intSegEnd):
                    end=None
                else:
                    end=(intSegEnd-fileS).seconds
                tmpData=loadDataOfPatient(indexPatient, fInfo.nameFile)
                if(not end==None):
                    end=end*256
                if(tmpData.shape[0]<22):
                    print(patients[indexPatient] +"  HAS FEWER CHANNELS")
                else:
                    interictalData=np.concatenate((interictalData, tmpData[0:22,start*256:end]), axis=1)
                    notUsed= createSpectrogram(interictalData)
                    totInst+=interictalData.shape[1]/256-notUsed/256
                    interictalData = np.delete(interictalData, np.s_[0:interictalData.shape[1]-notUsed], axis=1)

        # window_size:interictal_data_length=S:(preictal_data_length-30_SEC_PER_SEIZURE)
        S=(_SIZE_WINDOW_IN_SECONDS*(len(preictalInfo)*_MINUTES_OF_PREICTAL*60-_SIZE_WINDOW_IN_SECONDS*len(preictalInfo)))/totInst
        if(not (signalsBlock is None)):
            saveSignalsOnDisk(signalsBlock, nSpectogram)
        signalsBlock=None

        print("Spectrogram interictal: "+ str(nSpectogram))
        print("Hours interictal: " +str(totInst/60/60))
        legendOfOutput=str(nSpectogram)+"\\n"+legendOfOutput
        legendOfOutput="INTERICTAL"+"\\n"+legendOfOutput
        legendOfOutput="SEIZURE: " +str(len(preictalInfo))+"\\n"+legendOfOutput
        legendOfOutput=patients[indexPatient]+"\\n"+legendOfOutput
        allLegend=legendOfOutput
        legendOfOutput=''
        nSpectogram=0
        print("END creation interictal spectrogram")
        # END interictal data management loop

        # START preictal data management loop
        print("START creation preictal spectrogram")
        isPreictal='P'
        contSeizure=-1
        for pInfo in preictalInfo:
            contSeizure=contSeizure+1
            legendOfOutput=legendOfOutput+"SEIZURE "+str(contSeizure)+"\\n"
            preictalData = np.array([]).reshape(22,0)
            j=0
            for j in range(0,len(filesInfo)):
                if(pInfo.start>=filesInfo[j].start and pInfo.start<filesInfo[j].end):
                    break
            start=(pInfo.start-filesInfo[j].start).seconds
            if(start<0):
                start=0 # if preictal starts before file start
            end=None
            tmpData=[]
            if(pInfo.end<=filesInfo[j].end):
                end=(pInfo.end-filesInfo[j].start).seconds
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,start*256:end*256]), axis=1)
            else:
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,start*256:]), axis=1)
                end=(pInfo.end-filesInfo[j+1].start).seconds
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j+1].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,0:end*256]), axis=1)
            notUsed= createSpectrogram(preictalData, S=S)
            if(not (signalsBlock is None)):
                saveSignalsOnDisk(signalsBlock, nSpectogram)
            signalsBlock=None

        allLegend=allLegend+"\\n"+"PREICTAL"+"\\n"+str(nSpectogram)+"\\n"+legendOfOutput
        print("Spectrogram preictal: "+ str(nSpectogram))
        print("SEIZURE: " +str(len(preictalInfo)))
        print("END creation preictal spectrogram")
        # END preictal data management loop

        # START 'real' preictal data management loop
        print("START creation 'real' preictal spectrogram")
        isPreictal='P_R'
        nSpectogram=0
        contSeizure=-1
        S=0
        legendOfOutput=''
        for pInfo in preictalInfo:
            contSeizure=contSeizure+1
            legendOfOutput=legendOfOutput+"SEIZURE "+str(contSeizure)+"\\n"
            preictalData = np.array([]).reshape(22,0)
            j=0
            for j in range(0,len(filesInfo)):
                if(pInfo.start>=filesInfo[j].start and pInfo.start<filesInfo[j].end):
                    break
            start=(pInfo.start-filesInfo[j].start).seconds
            if(start<0):
                start=0 # if preictal starts before file start
            end=None
            tmpData=[]
            if(pInfo.end<=filesInfo[j].end):
                end=(pInfo.end-filesInfo[j].start).seconds
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,start*256:end*256]), axis=1)
            else:
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,start*256:]), axis=1)
                end=(pInfo.end-filesInfo[j+1].start).seconds
                tmpData=loadDataOfPatient(indexPatient, filesInfo[j+1].nameFile)
                preictalData=np.concatenate((preictalData, tmpData[0:22,0:end*256]), axis=1)
            notUsed= createSpectrogram(preictalData, S=S)
            if(not (signalsBlock is None)):
                saveSignalsOnDisk(signalsBlock, nSpectogram)
            signalsBlock=None

        allLegend=allLegend+"\\n"+"REAL_PREICTAL"+"\\n"+str(nSpectogram)+"\\n"+legendOfOutput
        print("Spectrogram 'REAL' preictal: "+ str(nSpectogram))
        print("END creation 'real' preictal spectrogram")
        # END preictal data management loop

        text_file = open(FirstPartPathOutput+SecondPartPathOutput+"/legendAllData.txt", "w")
        text_file.write(allLegend)
        text_file.close()
        print("Legend saved on disk")
        print('\\n')
    print("END")

In [ ]:
# Uncomment to run data processing
main_data_processing()

START \n
Parameters loaded
Working on patient 01
Summary patient loaded
START creation interictal spectrogram
Spectrogram interictal: 1710
Hours interictal: 14.49611111111111
END creation interictal spectrogram
START creation preictal spectrogram
Spectrogram preictal: 1704
SEIZURE: 7
END creation preictal spectrogram
START creation 'real' preictal spectrogram
Spectrogram 'REAL' preictal: 405
END creation 'real' preictal spectrogram
Legend saved on disk
\n
Working on patient 02
Summary patient loaded
START creation interictal spectrogram
Spectrogram interictal: 3099
Hours interictal: 26.263333333333332
END creation interictal spectrogram
START creation preictal spectrogram
Spectrogram preictal: 3156
SEIZURE: 3
END creation preictal spectrogram
START creation 'real' preictal spectrogram
Spectrogram 'REAL' preictal: 177
END creation 'real' preictal spectrogram
Legend saved on disk
\n
Working on patient 05
Summary patient loaded
START creation interictal spectrogram
Spectrogram interictal:

In [ ]:
!curl --upload-file ./spectograms https://transfer.sh

curl: (7) Failed to connect to transfer.sh port 443 after 155 ms: Connection refused


In [ ]:
# custom script for repearing the legendAllData.txt
import os

patients = ["01", "02", "05", "19", "21", "23"]

base_dir = "spectograms"

for p in patients:
    folder = os.path.join(base_dir, f"paz{p}")
    file_path = os.path.join(folder, "legendAllData.txt")

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    print(f"Repairing: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Replace literal "\n" with real newline
    repaired = content.replace("\\n", "\n")

    # Backup original file just in case
    backup_path = file_path + ".backup"
    if not os.path.exists(backup_path):
        with open(backup_path, "w", encoding="utf-8") as f:
            f.write(content)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(repaired)

print("Legend files repaired successfully.")

Repairing: spectograms\paz01\legendAllData.txt
Repairing: spectograms\paz02\legendAllData.txt
Repairing: spectograms\paz05\legendAllData.txt
Repairing: spectograms\paz19\legendAllData.txt
Repairing: spectograms\paz21\legendAllData.txt
Repairing: spectograms\paz23\legendAllData.txt
Legend files repaired successfully.


In [ ]:
!pip install huggingface_hub git-lfs

In [ ]:
!git lfs install

Git LFS initialized.


In [ ]:
!git clone https://huggingface.co/datasets/yussef-dev/seizure-spectrograms

Cloning into 'seizure-spectrograms'...
remote: Enumerating objects: 657, done.
remote: Total 657 (delta 0), reused 0 (delta 0), pack-reused 657 (from 1)
Receiving objects: 100% (657/657), 102.94 KiB | 4.12 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Filtering content: 100% (632/632), 33.00 GiB | 37.52 MiB/s, done.


---
## Part 2: CNN Model — Training and Evaluation

3D CNN for seizure prediction with leave-one-seizure-out cross-validation.

### CNN Imports

In [ ]:
#START code to train the network on the CPU
import os
#os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
#os.environ["CUDA_VISIBLE_DEVICES"] = ""
#END code to train the network on the CPU

import keras
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Conv3D, Dropout, Flatten, BatchNormalization
from keras.callbacks import EarlyStopping
from random import shuffle
import math
#to plot the model
#from keras.utils.vis_utils import plot_model
#from keras.models import load_model
# Returns a compiled model identical to the saved one
#model = load_model('my_model.h5')

### CNN Configuration

In [ ]:
PathSpectogramFolder = ''
OutputPath = ''
OutputPathModels = ''
interictalSpectograms = []
preictalSpectograms = []  # This array contains synthetic data, created to have a balanced dataset, used for training
preictalRealSpectograms = []  # This array contains the real preictal data, used for testing
patients = ["01", "02", "05", "19", "21", "23"]
nSeizure = 0

### Load CNN Parameters

In [ ]:
def loadParametersFromFile_CNN(filePath):
    global PathSpectogramFolder
    global OutputPath
    global OutputPathModels
    if(os.path.isfile(filePath)):
        with open(filePath, "r") as f:
                line=f.readline()
                if(line.split(":")[0]=="PathSpectogramFolder"):
                    PathSpectogramFolder=line.split(":")[1].strip()
                line=f.readline()
                if(line.split(":")[0]=="OutputPath"):
                    OutputPath=line.split(":")[1].strip()
                line=f.readline()
                if(line.split(":")[0]=="OutputPathModels"):
                    OutputPathModels=line.split(":")[1].strip()

### Load Spectrogram Data from Legend Files

In [ ]:
def loadSpectogramData(indexPat):
    global interictalSpectograms
    global preictalSpectograms
    global preictalRealSpectograms
    global nSeizure
    nFileForSeizure=0

    interictalSpectograms=[]
    preictalSpectograms=[]
    preictalRealSpectograms=[]

    f = open(PathSpectogramFolder+'/paz'+patients[indexPat]+'/legendAllData.txt', 'r')
    line=f.readline()
    while(not "SEIZURE" in line):
        line=f.readline()
    nSeizure=int(line.split(":")[1].strip())
    line=f.readline()
    line=f.readline()  # reads the number of spectrograms, not saved since not needed
    nSpectograms=int(line.strip())
    nFileForSeizure=math.ceil(math.ceil(nSpectograms/50)/nSeizure)
    line=f.readline()  # reads the path of the first file

    # Reading Interictal file paths
    cont=-1
    indFilePathRead=0
    while("npy" in line and indFilePathRead<nSeizure*nFileForSeizure):
        if(indFilePathRead%nFileForSeizure==0):
            interictalSpectograms.append([])
            cont=cont+1
            interictalSpectograms[cont].append(line.split(' ')[2].rstrip())
            indFilePathRead=indFilePathRead+1
        else:
            if(len(line.split(' '))>=3):
                interictalSpectograms[cont].append(line.split(' ')[2].rstrip())
            indFilePathRead=indFilePathRead+1

        line=f.readline()
    line=f.readline()  # reads PREICTAL
    line=f.readline()  # reads n spectograms
    line=f.readline()  # reads n seizure (SEIZURE X)

    # Reading Preictal file paths
    cont=-1
    indFilePathRead=0
    while(line.strip()!=""):
        if("SEIZURE" in line):
            line=f.readline()  # read n seizure (SEIZURE X) so advance forward
            if(len(line.split(' '))>=3):
                preictalSpectograms.append([])
                cont=cont+1
                preictalSpectograms[cont].append(line.split(' ')[2].rstrip())
                indFilePathRead=indFilePathRead+1
        else:
            if(len(line.split(' '))>=3):
                preictalSpectograms[cont].append(line.split(' ')[2].rstrip())
            indFilePathRead=indFilePathRead+1

        line=f.readline()

    line=f.readline()  # reads REAL_PREICTAL
    line=f.readline()  # reads n spectograms
    line=f.readline()  # reads n seizure (SEIZURE X)

    # Reading Real Preictal file paths
    cont=-1
    while(line):
        if("SEIZURE" in line):
            line=f.readline()  # read n seizure (SEIZURE X) so advance forward
            preictalRealSpectograms.append([])
            cont=cont+1
            preictalRealSpectograms[cont].append(line.split(' ')[2].rstrip())
        else:
            preictalRealSpectograms[cont].append(line.split(' ')[2].rstrip())

        line=f.readline()
    f.close()

### CNN Model Architecture

3-layer 3D CNN with BatchNormalization, Dropout, and Softmax output.

In [ ]:
def createModel():
    input_shape=(1, 22, 59, 114)
    model = Sequential()
    #C1
    model.add(Conv3D(16, (22, 5, 5), strides=(1, 2, 2), padding='valid',activation='relu',data_format= "channels_first", input_shape=input_shape))
    model.add(keras.layers.MaxPooling3D(pool_size=(1, 2, 2),data_format= "channels_first",  padding='same'))
    model.add(BatchNormalization())

    #C2
    model.add(Conv3D(32, (1, 3, 3), strides=(1, 1,1), padding='valid',data_format= "channels_first",  activation='relu'))  # uncertainty about whether to remove padding
    model.add(keras.layers.MaxPooling3D(pool_size=(1,2, 2),data_format= "channels_first", ))
    model.add(BatchNormalization())

    #C3
    model.add(Conv3D(64, (1,3, 3), strides=(1, 1,1), padding='valid',data_format= "channels_first",  activation='relu'))  # uncertainty about whether to remove padding
    model.add(keras.layers.MaxPooling3D(pool_size=(1,2, 2),data_format= "channels_first", ))
    model.add(BatchNormalization())

    model.add(Flatten())
    model.add(Dropout(0.5))
    model.add(Dense(256, activation='sigmoid'))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='softmax'))

    opt_adam = keras.optimizers.Adam(learning_rate=0.00001, beta_1=0.9, beta_2=0.999, epsilon=1e-08)
    model.compile(loss='categorical_crossentropy', optimizer=opt_adam, metrics=['accuracy'])

    return model

### Data Generators & Helper Functions

In [ ]:
def getFilesPathWithoutSeizure(indexSeizure, indexPat):
    filesPath=[]
    for i in range(0, nSeizure):
        if(i!=indexSeizure):
            filesPath.extend(interictalSpectograms[i])
            filesPath.extend(preictalSpectograms[i])
    shuffle(filesPath)
    return filesPath


def generate_arrays_for_training(indexPat, paths, start=0, end=100):
    while True:
        from_=int(len(paths)/100*start)
        to_=int(len(paths)/100*end)
        for i in range(from_, int(to_)):
            f=paths[i]
            x = np.load(PathSpectogramFolder+f)
            x=np.array([x])
            x=x.swapaxes(0,1)
            if('P' in f):
                y = np.repeat([[0,1]],x.shape[0], axis=0)
            else:
                y =np.repeat([[1,0]],x.shape[0], axis=0)
            yield(x,y)

def generate_arrays_for_predict(indexPat, paths, start=0, end=100):
    while True:
        from_=int(len(paths)/100*start)
        to_=int(len(paths)/100*end)
        for i in range(from_, int(to_)):
            f=paths[i]
            x = np.load(PathSpectogramFolder+f)
            x=np.array([x])
            x=x.swapaxes(0,1)
            #yield(x)
            yield(x,)

### Custom Early Stopping Callback

In [ ]:
class EarlyStoppingByLossVal(keras.callbacks.Callback):
    def __init__(self, monitor='val_loss', value=0.00001, verbose=0, lower=True):
        super(keras.callbacks.Callback, self).__init__()
        self.monitor = monitor
        self.value = value
        self.verbose = verbose
        self.lower=lower

    def on_epoch_end(self, epoch, logs={}):
        current = logs.get(self.monitor)
        if self.lower:
            if current < self.value:
                if self.verbose > 0:
                    print("Epoch %05d: early stopping THR" % epoch)
                self.model.stop_training = True
        else:
            if current > self.value:
                if self.verbose > 0:
                    print("Epoch %05d: early stopping THR" % epoch)
                self.model.stop_training = True

### CNN Training & Evaluation Pipeline

Leave-one-seizure-out cross-validation with threshold-based alarm evaluation.

In [ ]:
def main_cnn():
    print("START")
    loadParametersFromFile_CNN("PARAMETERS_CNN.txt")
    if not os.path.exists(OutputPathModels):
        os.makedirs(OutputPathModels)
    #callback=EarlyStopping(monitor='val_acc', min_delta=0, patience=0, verbose=0, mode='auto', baseline=None)
    callback=EarlyStoppingByLossVal(monitor='val_accuracy', value=0.975, verbose=1, lower=False)
    print("Parameters loaded")

    for indexPat in range(0, len(patients)):
        print('Patient '+patients[indexPat])
        if not os.path.exists(OutputPathModels+"ModelPat"+patients[indexPat]+"/"):
            os.makedirs(OutputPathModels+"ModelPat"+patients[indexPat]+"/")
        loadSpectogramData(indexPat)
        print('Spectograms data loaded')

        result='Patient '+patients[indexPat]+'\\n'
        result='Out Seizure, True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR \\n'
        for i in range(0, nSeizure):
            print('SEIZURE OUT: '+str(i+1))

            print('Training start')
            model = createModel()
            filesPath=getFilesPathWithoutSeizure(i, indexPat)

            #model.fit(generate_arrays_for_training(indexPat, filesPath, end=75),
            #                    validation_data=generate_arrays_for_training(indexPat, filesPath, start=75),
            #                    steps_per_epoch=int((len(filesPath)-int(len(filesPath)/100*25))),
            #                    validation_steps=int((len(filesPath)-int(len(filesPath)/100*75))),
            #                    verbose=2,
            #                    epochs=300, max_queue_size=2, shuffle=True, callbacks=[callback])  # 100 epochs is better #add stop criterion based on accuracy

            model.fit(generate_arrays_for_training(indexPat, filesPath, end=75),
                                validation_data=generate_arrays_for_training(indexPat, filesPath, start=75),
                                steps_per_epoch=int((len(filesPath)-int(len(filesPath)/100*25))),
                                validation_steps=int((len(filesPath)-int(len(filesPath)/100*75))),
                                verbose=2,
                                epochs=300, shuffle=True, callbacks=[callback])  # 100 epochs is better #add stop criterion based on accuracy
            print('Training end')

            print('Testing start')
            #filesPath=interictalSpectograms[i]
            #interPrediction=model.predict(generate_arrays_for_predict(indexPat, filesPath), max_queue_size=4, steps=len(filesPath))
            #filesPath=preictalRealSpectograms[i]
            #preictPrediction=model.predict(generate_arrays_for_predict(indexPat, filesPath), max_queue_size=4, steps=len(filesPath))

            filesPath=interictalSpectograms[i]
            interPrediction=model.predict(generate_arrays_for_predict(indexPat, filesPath), steps=len(filesPath))
            filesPath=preictalRealSpectograms[i]
            preictPrediction=model.predict(generate_arrays_for_predict(indexPat, filesPath), steps=len(filesPath))
            print('Testing end')


            # Creates a HDF5 file
            model.save(OutputPathModels+"ModelPat"+patients[indexPat]+"/"+'ModelOutSeizure'+str(i+1)+'.h5')
            print("Model saved")

            #to plot the model
            #plot_model(model, to_file="CNNModel", show_shapes=True, show_layer_names=True)

            if not os.path.exists(OutputPathModels+"OutputTest"+"/"):
                os.makedirs(OutputPathModels+"OutputTest"+"/")
            np.savetxt(OutputPathModels+"OutputTest"+"/"+"Int_"+patients[indexPat]+"_"+str(i+1)+".csv", interPrediction, delimiter=",")
            np.savetxt(OutputPathModels+"OutputTest"+"/"+"Pre_"+patients[indexPat]+"_"+str(i+1)+".csv", preictPrediction, delimiter=",")

            secondsInterictalInTest=len(interictalSpectograms[i])*50*30
            acc=0  # accumulator
            fp=0
            tp=0
            fn=0
            lastTenResult=list()

            for el in interPrediction:
                if(el[1]>0.5):
                    acc=acc+1
                    lastTenResult.append(1)
                else:
                    lastTenResult.append(0)
                if(len(lastTenResult)>10):
                    acc=acc-lastTenResult.pop(0)
                if(acc>=8):
                  fp=fp+1
                  lastTenResult=list()
                  acc=0

            lastTenResult=list()
            for el in preictPrediction:
                if(el[1]>0.5):
                    acc=acc+1
                    lastTenResult.append(1)
                else:
                    lastTenResult.append(0)
                if(len(lastTenResult)>10):
                    acc=acc-lastTenResult.pop(0)
                if(acc>=8):
                  tp=tp+1
                else:
                    if(len(lastTenResult)==10):
                       fn=fn+1

            sensitivity=tp/(tp+fn)
            FPR=fp/(secondsInterictalInTest/(60*60))

            result=result+str(i+1)+','+str(tp)+','+str(fp)+','+str(fn)+','+str(secondsInterictalInTest)+','
            result=result+str(sensitivity)+','+str(FPR)+'\\n'
            print('True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR')
            print(str(tp)+','+str(fp)+','+str(fn)+','+str(secondsInterictalInTest)+','+str(sensitivity)+','+str(FPR))
        with open(OutputPath, "a+") as myfile:
            myfile.write(result)

In [ ]:
# Uncomment to run CNN training
main_cnn()

START
Parameters loaded
Patient 01
Spectograms data loaded
SEIZURE OUT: 1
Training start


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/300
45/45 - 38s - 835ms/step - accuracy: 0.4982 - loss: 0.8675 - val_accuracy: 0.4667 - val_loss: 0.7549
Epoch 2/300
45/45 - 6s - 133ms/step - accuracy: 0.4716 - loss: 0.8867 - val_accuracy: 0.4667 - val_loss: 0.7393
Epoch 3/300
45/45 - 10s - 226ms/step - accuracy: 0.4927 - loss: 0.8516 - val_accuracy: 0.4667 - val_loss: 0.7290
Epoch 4/300
45/45 - 6s - 126ms/step - accuracy: 0.4963 - loss: 0.8470 - val_accuracy: 0.4667 - val_loss: 0.7215
Epoch 5/300
45/45 - 10s - 227ms/step - accuracy: 0.5146 - loss: 0.8354 - val_accuracy: 0.4667 - val_loss: 0.7131
Epoch 6/300
45/45 - 5s - 117ms/step - accuracy: 0.5389 - loss: 0.8053 - val_accuracy: 0.4667 - val_loss: 0.7064
Epoch 7/300
45/45 - 6s - 129ms/step - accuracy: 0.5156 - loss: 0.8365 - val_accuracy: 0.4667 - val_loss: 0.6973
Epoch 8/300
45/45 - 5s - 117ms/step - accuracy: 0.5247 - loss: 0.8029 - val_accuracy: 0.4667 - val_loss: 0.6861
Epoch 9/300
45/45 - 6s - 127ms/step - accuracy: 0.5224 - loss: 0.8082 - val_accuracy: 0.4707 - val_lo

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
54,6,0,7500,1.0,2.88
SEIZURE OUT: 2
Training start
Epoch 1/300
45/45 - 28s - 624ms/step - accuracy: 0.5005 - loss: 0.8640 - val_accuracy: 0.4540 - val_loss: 0.7683
Epoch 2/300
45/45 - 8s - 170ms/step - accuracy: 0.5131 - loss: 0.8338 - val_accuracy: 0.4540 - val_loss: 0.7437
Epoch 3/300
45/45 - 6s - 130ms/step - accuracy: 0.5068 - loss: 0.8299 - val_accuracy: 0.4540 - val_loss: 0.7258
Epoch 4/300
45/45 - 5s - 117ms/step - accuracy: 0.4882 - loss: 0.8604 - val_accuracy: 0.4540 - val_loss: 0.7137
Epoch 5/300
45/45 - 6s - 131ms/step - accuracy: 0.5009 - loss: 0.8533 - val_accuracy: 0.4513 - val_loss: 0.7052
Epoch 6/300
45/45 - 10s - 225ms/step - accuracy: 0.4968 - loss: 0.8361 - val_accuracy: 0.4102 - val_loss: 0.7007
Epoch 7/300
45/45 - 6s - 125ms/step - accuracy: 0.5023 - loss: 0.8385 - val_accuracy: 0.4979 - val_loss: 0.6952
Epoch 8/300
45/45 - 5s - 118ms/step - accuracy: 0.

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
52,0,0,7500,1.0,0.0
SEIZURE OUT: 3
Training start
Epoch 1/300
45/45 - 30s - 670ms/step - accuracy: 0.4888 - loss: 0.8670 - val_accuracy: 0.4011 - val_loss: 0.8108
Epoch 2/300
45/45 - 5s - 119ms/step - accuracy: 0.4810 - loss: 0.8624 - val_accuracy: 0.4011 - val_loss: 0.7807
Epoch 3/300
45/45 - 6s - 131ms/step - accuracy: 0.5116 - loss: 0.8275 - val_accuracy: 0.4011 - val_loss: 0.7589
Epoch 4/300
45/45 - 10s - 221ms/step - accuracy: 0.4902 - loss: 0.8587 - val_accuracy: 0.4011 - val_loss: 0.7379
Epoch 5/300
45/45 - 6s - 126ms/step - accuracy: 0.5002 - loss: 0.8348 - val_accuracy: 0.4011 - val_loss: 0.7206
Epoch 6/300
45/45 - 10s - 228ms/step - accuracy: 0.5185 - loss: 0.8124 - val_accuracy: 0.4011 - val_loss: 0.7050
Epoch 7/300
45/45 - 5s - 118ms/step - accuracy: 0.5144 - loss: 0.8341 - val_accuracy: 0.4037 - val_loss: 0.6951
Epoch 8/300
45/45 - 6s - 130ms/step - accuracy: 0.

...

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
52,0,0,7500,1.0,0.0
SEIZURE OUT: 5
Training start
Epoch 1/300
45/45 - 30s - 663ms/step - accuracy: 0.4961 - loss: 1.0914 - val_accuracy: 0.5989 - val_loss: 0.7590
Epoch 2/300
45/45 - 6s - 123ms/step - accuracy: 0.4920 - loss: 0.8972 - val_accuracy: 0.5989 - val_loss: 0.7379
Epoch 3/300
45/45 - 5s - 117ms/step - accuracy: 0.5103 - loss: 0.8498 - val_accuracy: 0.5989 - val_loss: 0.7128
Epoch 4/300
45/45 - 6s - 130ms/step - accuracy: 0.5263 - loss: 0.8276 - val_accuracy: 0.5989 - val_loss: 0.6913
Epoch 5/300
45/45 - 10s - 226ms/step - accuracy: 0.5071 - loss: 0.8463 - val_accuracy: 0.5989 - val_loss: 0.6662
Epoch 6/300
45/45 - 5s - 118ms/step - accuracy: 0.5153 - loss: 0.8198 - val_accuracy: 0.5989 - val_loss: 0.6410
Epoch 7/300
45/45 - 6s - 128ms/step - accuracy: 0.5340 - loss: 0.8088 - val_accuracy: 0.5989 - val_loss: 0.6197
Epoch 8/300
45/45 - 5s - 116ms/step - accuracy: 0.5

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
52,0,0,7500,1.0,0.0
SEIZURE OUT: 6
Training start
Epoch 1/300
45/45 - 23s - 501ms/step - accuracy: 0.4661 - loss: 0.8915 - val_accuracy: 0.6057 - val_loss: 0.6801
Epoch 2/300
45/45 - 6s - 129ms/step - accuracy: 0.5043 - loss: 0.8373 - val_accuracy: 0.5398 - val_loss: 0.6913
Epoch 3/300
45/45 - 6s - 133ms/step - accuracy: 0.4863 - loss: 0.8293 - val_accuracy: 0.5398 - val_loss: 0.6924
Epoch 4/300
45/45 - 10s - 220ms/step - accuracy: 0.5159 - loss: 0.8079 - val_accuracy: 0.4008 - val_loss: 0.6958
Epoch 5/300
45/45 - 6s - 135ms/step - accuracy: 0.4877 - loss: 0.8359 - val_accuracy: 0.4575 - val_loss: 0.6990
Epoch 6/300
45/45 - 5s - 120ms/step - accuracy: 0.4885 - loss: 0.8507 - val_accuracy: 0.4602 - val_loss: 0.6985
Epoch 7/300
45/45 - 6s - 132ms/step - accuracy: 0.4921 - loss: 0.8263 - val_accuracy: 0.4602 - val_loss: 0.6970
Epoch 8/300
45/45 - 10s - 228ms/step - accuracy: 0.

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
35,0,9,7500,0.7954545454545454,0.0
SEIZURE OUT: 7
Training start
Epoch 1/300
45/45 - 29s - 649ms/step - accuracy: 0.4937 - loss: 0.8599 - val_accuracy: 0.6048 - val_loss: 0.6720
Epoch 2/300
45/45 - 18s - 405ms/step - accuracy: 0.4932 - loss: 0.8626 - val_accuracy: 0.6048 - val_loss: 0.6726
Epoch 3/300
45/45 - 7s - 163ms/step - accuracy: 0.5068 - loss: 0.8428 - val_accuracy: 0.6048 - val_loss: 0.6725
Epoch 4/300
45/45 - 6s - 127ms/step - accuracy: 0.5118 - loss: 0.8316 - val_accuracy: 0.6048 - val_loss: 0.6745
Epoch 5/300
45/45 - 6s - 132ms/step - accuracy: 0.5140 - loss: 0.8294 - val_accuracy: 0.6048 - val_loss: 0.6763
Epoch 6/300
45/45 - 5s - 121ms/step - accuracy: 0.5226 - loss: 0.8128 - val_accuracy: 0.6048 - val_loss: 0.6800
Epoch 7/300
45/45 - 10s - 226ms/step - accuracy: 0.5081 - loss: 0.8184 - val_accuracy: 0.6048 - val_loss: 0.6844
Epoch 8/300
45/45 - 5s - 122ms/step

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
9,0,41,7500,0.18,0.0
Patient 02
Spectograms data loaded
SEIZURE OUT: 1
Training start
Epoch 1/300
64/64 - 38s - 598ms/step - accuracy: 0.5054 - loss: 0.8844 - val_accuracy: 0.4748 - val_loss: 0.7123
Epoch 2/300
64/64 - 26s - 407ms/step - accuracy: 0.4870 - loss: 0.8995 - val_accuracy: 0.4748 - val_loss: 0.7084
Epoch 3/300
64/64 - 26s - 413ms/step - accuracy: 0.4981 - loss: 0.8725 - val_accuracy: 0.4748 - val_loss: 0.7028
Epoch 4/300
64/64 - 26s - 407ms/step - accuracy: 0.5241 - loss: 0.8328 - val_accuracy: 0.4748 - val_loss: 0.6883
Epoch 5/300
64/64 - 27s - 417ms/step - accuracy: 0.5311 - loss: 0.8296 - val_accuracy: 0.5176 - val_loss: 0.6671
Epoch 6/300
64/64 - 26s - 409ms/step - accuracy: 0.5403 - loss: 0.8059 - val_accuracy: 0.7507 - val_loss: 0.6287
Epoch 7/300
64/64 - 26s - 411ms/step - accuracy: 0.5555 - loss: 0.7950 - val_accuracy: 0.7517 - val_loss: 0.6082
Epoch 8/30

Testing end
Model saved
True Positive, False Positive, False negative, Second of Inter in Test, Sensitivity, FPR
52,111,0,31500,1.0,12.685714285714285
SEIZURE OUT: 2
Training start
Epoch 1/300
64/64 - 42s - 663ms/step - accuracy: 0.5108 - loss: 0.8823 - val_accuracy: 0.5000 - val_loss: 0.7457
Epoch 2/300
64/64 - 26s - 400ms/step - accuracy: 0.4940 - loss: 0.8736 - val_accuracy: 0.5000 - val_loss: 0.7104
Epoch 3/300
64/64 - 25s - 396ms/step - accuracy: 0.4857 - loss: 0.8529 - val_accuracy: 0.5000 - val_loss: 0.6917
Epoch 4/300
64/64 - 25s - 395ms/step - accuracy: 0.5047 - loss: 0.8435 - val_accuracy: 0.6609 - val_loss: 0.6833
Epoch 5/300
64/64 - 25s - 396ms/step - accuracy: 0.5089 - loss: 0.8358 - val_accuracy: 0.6336 - val_loss: 0.6746
Epoch 6/300
64/64 - 26s - 402ms/step - accuracy: 0.5205 - loss: 0.8218 - val_accuracy: 0.6545 - val_loss: 0.6681
Epoch 7/300
64/64 - 25s - 395ms/step - accuracy: 0.5285 - loss: 0.8193 - val_accuracy: 0.6518 - val_loss: 0.6626
Epoch 8/300
64/64 - 26s - 40

---
## Part 3: Threshold Testing

Evaluates average and standard deviation of sensitivity and FPR across different thresholds per patient.

### Threshold Configuration & Evaluation

In [ ]:
import statistics

OutputPathModels_test = ['**********/PrimaEsecuzione/',
                         '***********/SecondaEsecuzione/']  # path of folder containing the output files of the model in the test phase

# This evaluates the average and the standard deviation of the sensitivity and FPR for
# the output of the network in the test phase. It's used to change the threshold
# and adapt it to each patient.
def main_threshold():
    pat=["01","02", "05", "19","21", "23"]
    nSeizure=[7,3, 5, 2, 4,5]
    secondsInterictalInTest=[7500,31500, 10500, 46500, 21000,10500]
    threshold=[0.6,0.8, 0.4, 0.001, 0.3,0.3]
    #totSens=0
    #totFPR=0
    for j in range(0,len(pat)):
        sensResults=[]
        FPRResults=[]
        for k in range(0,2):
            for i in range(0, nSeizure[j]):
                interPrediction=np.loadtxt(OutputPathModels_test[k]+"OutputTest"+"/"+"Int_"+pat[j]+"_"+str(i+1)+".csv",delimiter=',')
                preictPrediction=np.loadtxt(OutputPathModels_test[k]+"OutputTest"+"/"+"Pre_"+pat[j]+"_"+str(i+1)+".csv",delimiter=',')

                acc=0  # accumulator
                fp=0
                tp=0
                fn=0
                lastTenResult=list()

                for el in interPrediction:
                    if(el[1]>threshold[j]):
                        acc=acc+1
                        lastTenResult.append(1)
                    else:
                        lastTenResult.append(0)
                    if(len(lastTenResult)>10):
                        acc=acc-lastTenResult.pop(0)
                    if(acc>=8):
                      fp=fp+1
                      lastTenResult=list()
                      acc=0

                lastTenResult=list()
                for el in preictPrediction:
                    if(el[1]>threshold[j]):
                        acc=acc+1
                        lastTenResult.append(1)
                    else:
                        lastTenResult.append(0)
                    if(len(lastTenResult)>10):
                        acc=acc-lastTenResult.pop(0)
                    if(acc>=8):
                      tp=tp+1
                    else:
                        if(len(lastTenResult)==10):
                           fn=fn+1

                sensitivity=tp/(tp+fn)*100
                FPR=fp/(secondsInterictalInTest[j]/(60*60))
                sensResults.append(sensitivity)
                FPRResults.append(FPR)

        sdSENS=statistics.stdev(sensResults)
        avSENS=statistics.mean(sensResults)

        sdFPR=statistics.stdev(FPRResults)
        avFPR=statistics.mean(FPRResults)
        print(pat[j]+"   AVG_Sens= "+str(avSENS)+" +- "+str(sdSENS)+"   AVG_FPR= "+str(avFPR)+" +- "+str(sdFPR))

In [ ]:
# Uncomment to run threshold testing
# main_threshold()